# Reinforcement learning

Plan for reinforcement learning:
- Setup with Farama's Gymnasium environment. 
- Define the environment rules - when to end the episode, what would be the reward, action.
- Build and train the RL model
- Evaluate the performance of the model 
Reward function:
- The estimates are: Rise, Fall, Neutral
- Reward of +0 going to state ...
- Reward of +1 going to state ...
- Reward of -1 going to state ...


## Data preparation

Even though I preprocessed my dataset with normalized data values, I again needed change the shape of the data for RL training. It is because that the model learns better with context. As described in my previous notebook, the game playing DQN models use at least 3 consecutive frames by replacing the RGB channels in the input data. That enabled the model to learn changes or motions within the input data to predict the right actions. 

In [ ]:
# First a function to read a file
import pandas as pd
from pathlib import Path
import os
from tqdm import tqdm

# The folder will be provided as an argument and the function will return the file names
def load_file_names(folder: str) -> pd.DataFrame:

    files = Path(folder).glob("*.parquet")
    # Just to make sure, it returns the same files for both training and validation data
    return [(Path('train_dataset')/file.name, Path('validation_dataset')/file.name) for file in files]

In [28]:
file_names = load_file_names("train_dataset")

In [37]:

display(file_names[1001])

print("🌮 Train path:", file_names[1001][0])
print("🌮 Test path:", file_names[1001][1])

(PosixPath('train_dataset/FENC_processed.parquet'),
 PosixPath('validation_dataset/FENC_processed.parquet'))

🌮 Train path: train_dataset/FENC_processed.parquet
🌮 Test path: validation_dataset/FENC_processed.parquet


Data loading function is implemented below. The function will receive the file path and then return the lists of observations. I will use this function within the environment to load the training and evaluation data for each episodes. 

In [39]:
import numpy as np
    
def load_data(file_path: str, window: int = 60) -> tuple[np.ndarray, np.ndarray]:
    features = ['RSI_norm', 'MACD_line_norm', 'MACD_signal_norm', 'Volume_norm', 'Return']
    X = []
    y = []
    try:
        data = pd.read_parquet(file_path, engine='pyarrow')
        # If the data has just a little more than a year of data, I should skip it. 
        # I don't believe such small data will be useful after eliminating the first 60 days. 
        if len(data) < 300:
            return None, None
        for i in range(len(data) - window):
            X.append(data[features].iloc[i:i+window].values)
            # The value should be the 60th period values as I am wrapping up previous 60 days of data.
            # Corrected below to 1 less index to select the correct data
            y.append({
                'target': data['Target'].iloc[i+window-1], 
                'date': data.index[i+window-1], 
                'close': data['Close'].iloc[i+window-1]})
        return np.array(X), np.array(y)
    except Exception as e:
        print(f"An error occurred while importing data: {e}")
        return None, None
    

In [40]:
random_data, random_target = load_data(file_names[333][0])
print("\n🌸 Shape of random data:", random_data.shape if random_data is not None else "Not Enough data available!")
print("\nSample of the data:", "\n🌼 RSI: ",random_data[0][0][0], "\n🌼 MACD_line: ", random_data[0][0][1], "\n🌼 MACD_signal: ", random_data[0][0][2], "\n🌼 Volume: ", random_data[0][0][3], "\n🌼 Return: ", random_data[0][0][4])


🌸 Shape of random data: (503, 60, 5)

Sample of the data: 
🌼 RSI:  0.14446632136706852 
🌼 MACD_line:  0.9258362488525782 
🌼 MACD_signal:  0.4760976753947792 
🌼 Volume:  -0.03611208232927794 
🌼 Return:  0.0029513731957795386


## Reinforcement Learning 


In [5]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
def std_scaler(data, window=60):
    # Default window is 12 weeks of data (5 trading days per week)
    mean = data.rolling(window=window).mean()
    std_dev = data.rolling(window=window).std()
    z_score = (data - mean) / std_dev
    return z_score

In [ ]:
class MarketEnv(gym.Env):

    def __init__(self):
        # Observation space includes 12 dimensions data for technical indicators
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(12,), dtype=np.float32)

        # Prediction is a discrete action: 0 = hold, 1 = buy, 2 = sell
        self.action_space = gym.spaces.Discrete(3)

        # Score to track total rewards 
        self.total_score = 0.0

        # Map action numbers to actual movements on the grid
        # This makes the code more readable than using raw numbers
        self._action_to_direction = {
            0: np.array([0, 1]),   # Move right (column + 1)
            1: np.array([-1, 0]),  # Move up (row - 1)
            2: np.array([0, -1]),  # Move left (column - 1)
        }
        self.reset()

    def _get_obs(self):
        """Convert internal state to observation format.

        Returns:
            dict: Observation with agent and target positions
        """
        return {"score": self.total_score}
         # Calculate the observation vector for the current step
        obs = np.array([
            self.data['Adj Close'][self.current_step],
            # try yourself
            # get the current observation for each indicator
            # write your code here
             self.data['EMA7'][self.current_step],
            self.data['EMA14'][self.current_step],
            self.data['EMA50'][self.current_step],
            self.data['EMA200'][self.current_step],
            self.data['MACD_line'][self.current_step],
            self.data['MACD_signal'][self.current_step],
            self.data['MACD_diff'][self.current_step],
            self.data['RSI'][self.current_step],
            self.data['OBV'][self.current_step],
            self.data['BBH'][self.current_step],
            self.data['BBL'][self.current_step],
        ])

        # Return the observation vector
        return obs
    
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Start a new episode.

        Args:
            seed: Random seed for reproducible episodes
            options: Additional configuration (unused in this example)

        Returns:
            tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)

        # Randomly place the agent anywhere on the grid
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        # Randomly place target, ensuring it's different from agent position
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(
                0, self.size, size=2, dtype=int
            )

        observation = self._get_obs()
        info = self._get_info()

        return observation, info
    
    def step(self, action):
        """Execute one timestep within the environment.

        Args:
            action: The action to take (0-3 for directions)

        Returns:
            tuple: (observation, reward, terminated, truncated, info)
        """
        # Map the discrete action (0-3) to a movement direction
        direction = self._action_to_direction[action]

        # Update agent position, ensuring it stays within grid bounds
        # np.clip prevents the agent from walking off the edge
        self._agent_location = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )

        # Check if agent reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)

        # We don't use truncation in this simple environment
        # (could add a step limit here if desired)
        truncated = False

        # Simple reward structure: +1 for reaching target, 0 otherwise
        # Alternative: could give small negative rewards for each step to encourage efficiency
        distance = np.linalg.norm(self._agent_location - self._target_location)
        reward = 1 if terminated else -0.1 * distance   

        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, truncated, info

In [4]:
print("Current Accelerator:", torch.accelerator.current_accelerator())
# if torch.accelerator.is_available():
#     tensor = tensor.to(torch.accelerator.current_accelerator())

Current Accelerator: mps


In [ ]:
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

### Backtesting
To assess the performance by exploring how it would have performed using historical data.